In [1]:
import os
import duckdb 
import pandas as pd

os.getcwd()

'C:\\Users\\Hp\\1 Year Road Map for SQL Mastery'

In [2]:
con = duckdb.connect(
r"D:\DataQuartz\Kaggle Dataset Practice\Dataset 2\Instacart Market Basket Analysis\Instacart.duckdb"
)

In [3]:
con.execute("SHOW TABLES").df()

,name
0,aisles
1,departments
2,order_products
3,orders
4,products


# Step 1.1 — Orders Table Preview

## Objective

Understand what information exists at the order level.

In [4]:
con.execute(""" SELECT * FROM orders LIMIT 5""").df()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,08,NaN
1,2398795,1,prior,2,3,07,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,07,29.0
4,431534,1,prior,5,4,15,28.0


## Observation

The orders table contains one row per order placed by a customer. The `order_id` uniquely identifies each order, while `user_id` identifies the customer who placed it. The `order_number` shows the sequence of orders for a customer, allowing us to track customer purchasing history over time. The `order_dow` and `order_hour_of_day` columns provide information about when orders were placed, which can later be used for behavioral analysis. The `days_since_prior_order` column measures the gap between consecutive purchases and will be useful when analyzing customer reorder patterns and purchase frequency.

# Step 1.2 — Products Table Preview

## Objective

Understand product-level information and how products are categorized within the Instacart dataset.

In [5]:
con.execute(""" SELECT * FROM products LIMIT 5""").df()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


## Observation

The products table contains one row per product available in the Instacart catalog. Each product has a unique identifier and descriptive product name. Products are organized into a hierarchy through the `aisle_id` and `department_id` columns, allowing analysis to be performed at both detailed and aggregated category levels. This table will act as the primary product dimension throughout the project and will frequently be joined with transactional purchase data.

# Step 1.3 — Aisles Table Preview

## Objective

Understand how products are grouped into aisles within the Instacart catalog.

In [12]:
con.execute("""SELECT * FROM aisles LIMIT 5""").df()

,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


## Observation

The aisles table serves as a product categorization layer by grouping similar products into specific aisle categories. Each aisle represents a more detailed classification than a department, allowing products to be analyzed at a granular level. This table will commonly be joined with the products table to understand purchasing patterns across different product categories and customer preferences.

# Step 1.4 — Departments Table Preview

## Objective

Understand the highest-level product categorization structure within the Instacart dataset.

In [13]:
con.execute("""SELECT * FROM departments LIMIT 5""").df()

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


## Observation

The departments table provides the broadest level of product classification in the dataset. While aisles group products into more specific categories, departments bring those categories together under larger business groups such as produce, bakery, frozen, and alcohol. This structure will make it easier to roll up product analysis from detailed aisle-level insights to a higher-level department view when comparing customer purchasing behavior.

# Step 1.5 — Order Products Table Preview

## Objective

Understand how purchased products are linked to customer orders.

In [15]:
con.execute("""SELECT * FROM order_products LIMIT 5""").df()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


## Observation

The order_products table records the actual items purchased within each order. An order can contain multiple products, which is why the same order_id appears across several rows. The add_to_cart_order column captures the sequence in which products were added to the basket, while reordered indicates whether the customer had purchased that product before. This table acts as the bridge between customer orders and products, making it one of the most important tables for behavioral and purchasing analysis.

# Step 2.1 — Total Orders Analysis

## Objective

Determine the total number of orders available in the Instacart dataset.

In [16]:
con.execute("""SELECT COUNT(*) as Total_Orders
                FROM orders""").df()

,Total_Orders
0,3421083


## Observation

The dataset contains 3.4 million orders, providing a large sample of customer purchasing activity. At this scale, even simple aggregations become meaningful because they represent behavior across a substantial customer base. Understanding the overall order volume provides useful context before exploring customer patterns, product demand, and reorder behavior in later steps.

# Step 2.2 — Unique Customers Analysis

## Objective

Determine how many individual customers are represented in the dataset.

In [17]:
### Data Preview — order table

con.execute("""SELECT * FROM orders LIMIT 5""").df()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,08,NaN
1,2398795,1,prior,2,3,07,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,07,29.0
4,431534,1,prior,5,4,15,28.0


In [24]:
con.execute("""SELECT COUNT(DISTINCT user_id) as Unique_Customers
                FROM orders
                
                """).df()

,Unique_Customers
0,206209


## Observation

The dataset contains 206,209 unique customers. Comparing this figure with the total number of orders shows that most customers placed multiple orders rather than making a single purchase. This distinction between order volume and customer count provides an early indication of repeat purchasing behavior within the dataset.

# Step 2.3 — Average Orders per Customer

## Objective

Measure how frequently customers place orders by calculating the average number of orders per customer.

In [30]:
con.execute("""SELECT COUNT(*) / COUNT(DISTINCT user_id) as AVG_order_per_Cust
                FROM orders
                """).df()

,AVG_order_per_Cust
0,16.590367


## Observation

Customers place approximately 16.59 orders on average across the dataset. This highlights that the typical customer makes repeat purchases rather than placing a single order and leaving. The result provides an early indication of strong customer retention and suggests that analyzing reorder behavior could reveal valuable patterns in long-term purchasing habits.

# Step 2.4 — Order Distribution by Evaluation Set

## Objective

Understand how the dataset is divided across the available evaluation groups.

In [31]:
con.execute("""SELECT COUNT(*) as Total_Orders, eval_set
                FROM orders
                GROUP BY eval_set""").df()

,Total_Orders,eval_set
0,131209,train
1,3214874,prior
2,75000,test


## Observation

The dataset is primarily composed of prior orders, which represent historical customer purchases. A smaller portion is allocated to the train and test groups, which are commonly used for predictive modeling tasks. Understanding this distribution is important because most exploratory analysis will be performed on historical purchasing behavior, while the train and test sets support reorder prediction exercises.

# Step 2.5 — Order Activity by Day of Week

## Objective

Identify how customer ordering activity is distributed across the days of the week.

In [32]:
### Data Preview — order table

con.execute("""SELECT * FROM orders LIMIT 5""").df()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,08,NaN
1,2398795,1,prior,2,3,07,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,07,29.0
4,431534,1,prior,5,4,15,28.0


In [38]:
con.execute("""SELECT COUNT(*) as Total_Orders, order_dow
                FROM orders
                GROUP BY order_dow
                """).df()

,Total_Orders,order_dow
0,600905,0
1,587478,1
2,467260,2
3,436972,3
4,426339,4
5,453368,5
6,448761,6


## Observation

Order activity is not evenly distributed throughout the week. The highest order volumes occur on days 0 and 1, while midweek activity is noticeably lower. This pattern suggests that customers tend to place more orders around specific periods of the week, making day-of-week an important factor when analyzing purchasing behavior and customer engagement.

# Step 2.6 — Peak Ordering Hours

## Objective

Analyze how order volume changes throughout the day and identify the busiest ordering hours.

In [40]:
con.execute("""SELECT order_hour_of_day, COUNT(*) AS Total_Orders
                FROM orders
                GROUP BY order_hour_of_day
                ORDER BY order_hour_of_day""").df()

,order_hour_of_day,Total_Orders
0,00,22758
1,01,12398
2,02,7539
3,03,5474
4,04,5527
5,05,9569
6,06,30529
7,07,91868
8,08,178201
9,09,257812


## Observation

Order activity is concentrated during daytime hours, with volume increasing rapidly in the morning and remaining high through the early afternoon. Very few orders are placed overnight, while peak activity occurs between late morning and mid-afternoon. This pattern suggests that customers primarily use Instacart during regular daily routines rather than during late-night hours.

# Step 2.7 — Most Active Customers

## Objective

Identify the customers who have placed the highest number of orders in the dataset.

In [43]:
con.execute("""SELECT user_id, COUNT(*) AS Highest_Orders
                FROM orders
                GROUP BY user_id
                ORDER BY Highest_Orders DESC
                LIMIT 10""").df()

,user_id,Highest_Orders
0,34000,100
1,23405,100
2,26092,100
3,35662,100
4,123066,100
5,118218,100
6,126537,100
7,31118,100
8,22337,100
9,126848,100


## Observation

Several customers reached 100 total orders, indicating a highly engaged group of repeat buyers within the dataset. While the average customer places around 17 orders, these top customers demonstrate significantly higher purchasing activity and may represent loyal or long-term users of the platform. Identifying these high-frequency customers can be useful when analyzing retention, customer lifetime value, and purchasing habits.

# Step 2.8 — Customer Order Range

## Objective

Determine the minimum and maximum number of orders placed by any customer in the dataset.

In [49]:
con.execute("""SELECT MAX(Orders) as Max_Orders, MIN(Orders) as Min_Orders

FROM (

SELECT user_id, COUNT(*) AS Orders
                FROM orders
                GROUP BY user_id
                )

                
                
                """).df()

,Max_Orders,Min_Orders
0,100,4


## Observation

Customer ordering behavior varies considerably across the dataset. The most active customers placed up to 100 orders, while the least active customers placed only 4 orders. This range highlights the presence of both highly engaged repeat buyers and relatively infrequent shoppers, indicating different levels of customer loyalty and purchasing activity.

# Step 2.9 — Missing Values (NULL) Audit

## Objective

Identify whether any important columns in the orders table contain missing values.

In [50]:
### Data Preview — order table

con.execute("""SELECT * FROM orders LIMIT 5""").df()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,08,NaN
1,2398795,1,prior,2,3,07,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,07,29.0
4,431534,1,prior,5,4,15,28.0


In [55]:
con.execute("""SELECT COUNT(order_id), COUNT(*) AS Total_Orders,
                COUNT(user_id), COUNT(*) AS Total_Users,
                count(order_number), COUNT(*) AS Total_Order_No,
                COUNT(order_dow), COUNT(*) AS Total_Order_dow,
                COUNT(order_hour_of_day), COUNT(*) AS Total_ohod,
               COUNT(days_since_prior_order), COUNT(*) AS Total_dspo
                FROM orders
                
                """).df()

,count(order_id),Total_Orders,count(user_id),Total_Users,count(order_number),Total_Order_No,count(order_dow),Total_Order_dow,count(order_hour_of_day),Total_ohod,count(days_since_prior_order),Total_dspo
0,3421083,3421083,3421083,3421083,3421083,3421083,3421083,3421083,3421083,3421083,3214874,3421083


## Observation

Most columns in the orders table are fully populated, with no missing values detected in user_id, order_number, order_dow, or order_hour_of_day. The only column containing missing values is days_since_prior_order, which is expected because a customer's first recorded order has no previous order from which a time difference can be calculated. These NULL values represent valid business scenarios rather than data quality issues.

# Step 2.10 — Duplicate Order ID Check

## Objective

Verify that each order_id appears only once in the orders table.

In [57]:
con.execute("""SELECT COUNT(DISTINCT order_id), COUNT(*)
                FROM orders""").df()

,count(DISTINCT order_id),count_star()
0,3421083,3421083


# Step 2.11 — Duplicate Customer Check

## Objective

Determine whether customers can appear multiple times within the orders table.

In [59]:
con.execute("""SELECT COUNT(DISTINCT user_id), COUNT(*)
                FROM orders""").df()

,count(DISTINCT user_id),count_star()
0,206209,3421083


## Observation

Customer IDs appear multiple times throughout the orders table because each customer can place many orders over time. While there are 206,209 unique customers, the dataset contains over 3.4 million order records. This repetition is expected and reflects the one-to-many relationship between customers and orders rather than a data quality issue.

# Step 2.12 — First Order Identification

## Objective

Determine how many first-time orders exist in the dataset.

In [61]:
con.execute("""SELECT COUNT(user_id), COUNT(order_number) as First_Time_Orders
                FROM orders
                WHERE order_number = 1
                 """).df()
                

,count(user_id),First_Time_Orders
0,206209,206209


## Observation

The dataset contains 206,209 first-time orders, which matches the total number of unique customers identified earlier. This confirms that each customer has exactly one first recorded order, indicating that the customer ordering history is complete and logically consistent.

# Step 2.13 — Repeat Order Identification

## Objective

Determine how many orders were placed by returning customers.

In [62]:
con.execute("""SELECT COUNT(user_id), COUNT(order_number) as Repeat_Orders
                FROM orders
                WHERE order_number > 1""").df()

,count(user_id),Repeat_Orders
0,3214874,3214874


## Observation

The dataset contains 3,214,874 repeat orders, representing orders placed after a customer's first purchase. Compared to the 206,209 first-time orders identified earlier, this indicates that the majority of activity in the dataset comes from returning customers, making it well suited for customer behavior and retention analysis.

# Step 2.14 — Missing Days Since Prior Order Check

## Objective

Determine whether the `days_since_prior_order` column contains missing values.

In [63]:
con.execute("""SELECT COUNT(days_since_prior_order), COUNT(*)
                FROM orders""").df()

,count(days_since_prior_order),count_star()
0,3214874,3421083


## Observation

The `days_since_prior_order` column contains 206,209 missing values. This matches the number of first-time orders identified earlier in the analysis. Since a customer's first purchase has no previous order to compare against, these missing values are expected and do not indicate a data quality issue.

# Step 2.15 — Missing User ID Check

## Objective

Determine whether the `user_id` column contains missing values.

In [64]:
con.execute("""SELECT COUNT(user_id), COUNT(*)
                FROM orders""").df()

,count(user_id),count_star()
0,3421083,3421083


## Observation

The `user_id` column contains no missing values. The count of non-null user IDs matches the total number of rows in the orders table, confirming that every order is associated with a customer.

# Step 2.16 — Missing Order Hour Check

## Objective

Determine whether the `order_hour_of_day` column contains missing values.

In [65]:
con.execute("""SELECT COUNT(order_hour_of_day), COUNT(*)
                FROM orders""").df()

,count(order_hour_of_day),count_star()
0,3421083,3421083


## Observation

The `order_hour_of_day` column contains no missing values. Every order in the dataset has an associated purchase hour, allowing time-based analysis to be performed without requiring additional data cleaning.

# Step 2.17 — Customer With Most Orders

## Objective

Identify the customer who has placed the highest number of orders in the dataset.

In [75]:
con.execute("""SELECT user_id, COUNT(order_id) as MAX_Orders
                FROM orders
                GROUP BY user_id
                ORDER BY MAX_Orders DESC
                LIMIT 1
                """).df()

,user_id,MAX_Orders
0,69182,100


## Observation

The highest number of orders placed by a single customer is 100. Multiple customers reached this value, indicating that a small group of highly active customers placed significantly more orders than the average customer in the dataset.

# Step 2.18 — Customer With Fewest Orders

## Objective

Identify the customer who has placed the lowest number of orders in the dataset.

In [77]:
con.execute("""SELECT user_id, COUNT(order_id) as Total_Orders
                FROM orders
                GROUP BY user_id
                ORDER BY Total_Orders
                LIMIT 1
                """).df()

,user_id,Total_Orders
0,513,4


## Observation

The lowest number of orders placed by a customer is 4. This suggests that every customer in the dataset completed at least four purchases, while some customers placed significantly more orders, highlighting a wide range of customer activity levels.

## Observation

The customer with the highest activity placed 100 orders, while the least active customer placed 4 orders. This indicates a wide range of customer engagement levels within the dataset.

# Step 2.19 — Dataset Summary

## Objective

Create a final summary of the orders dataset by reviewing the key findings from the exploratory analysis.

## Task

Create a single query that returns:

- Total Orders
- Unique Customers
- Average Orders Per Customer

in one result table.

In [87]:
con.execute("""SELECT COUNT(order_id) AS Total_Orders, COUNT(DISTINCT user_id) as Unique_Customers,
                COUNT(order_id) / COUNT(DISTINCT user_id) AS AVG_Orders_Per_Customer
                FROM orders""").df()
                

,Total_Orders,Unique_Customers,AVG_Orders_Per_Customer
0,3421083,206209,16.590367


## Observation

The dataset contains 3,421,083 total orders from 206,209 unique customers. On average, each customer placed approximately 16.59 orders, indicating a strong level of repeat purchasing behavior across the customer base.

# Step 3.1 — Product Catalog Overview

## Objective

Understand the size of the product catalog available within the Instacart dataset.

In [14]:
### Data Preview — products table

con.execute(""" SELECT * FROM products LIMIT 5""").df()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [6]:
con.execute("""SELECT COUNT(*) AS Total_Products
                FROM products""").df()

,Total_Products
0,49688


## Observation

The products table contains 49,688 unique products. This represents the complete product catalog available for customer purchases within the dataset.

# Step 3.2 — Product Uniqueness Check

## Objective

Verify that each product appears only once in the product catalog.

In [8]:
con.execute("""SELECT COUNT(DISTINCT product_id) AS Unique_Products, COUNT(*) AS Total_Products
                FROM products""").df()

,Unique_Products,Total_Products
0,49688,49688


## Observation

The products table contains 49,688 unique products and 49,688 total rows.

Since the number of unique product IDs matches the total number of rows, every product_id appears exactly once in the products table. No duplicate product IDs exist.

# Step 3.3 — Missing Product ID Check

## Objective

Determine whether the product_id column contains missing values.

In [9]:
con.execute("""SELECT COUNT(product_id), COUNT(*) 
                FROM products""").df()

,count(product_id),count_star()
0,49688,49688


## Observation

The products table contains 49,688 rows and 49,688 non-null product_id values.

Since the count of product_id matches the total row count, no missing values exist in the product_id column.

# Step 3.4 — Missing Product Name Check

## Objective

Determine whether the product_name column contains missing values.

In [10]:
con.execute("""SELECT COUNT(product_name), COUNT(*)
                FROM products""").df()

,count(product_name),count_star()
0,49688,49688


## Observation

The products table contains 49,688 rows and 49,688 non-null product_name values.

Since the count of product_name matches the total row count, no missing values exist in the product_name column.

In [11]:
### Data Preview — aisles table

con.execute("""SELECT * FROM aisles LIMIT 5""").df()

,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


# Step 3.5 — Missing Aisle ID Check

## Objective

Determine whether the aisle_id column contains missing values.

In [12]:
con.execute("""SELECT COUNT(aisle_id) as Total_Aisles, COUNT(*)
                FROM aisles""").df()

,Total_Aisles,count_star()
0,134,134


## Observation

The aisles table contains 134 rows and 134 non-null aisle_id values.

Since the count of aisle_id matches the total row count, no missing values exist in the aisle_id column.

# Step 3.6 — Missing Department ID Check

## Objective

Determine whether the department_id column contains missing values.

In [15]:
### Data Preview — departments table

con.execute("""SELECT * FROM departments LIMIT 5""").df()

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


In [17]:
con.execute("""SELECT COUNT(department_id), COUNT(*)
                FROM departments""").df()

,count(department_id),count_star()
0,21,21


## Observation

The departments table contains 21 rows and 21 non-null department_id values.

Since the count of department_id matches the total row count, no missing values exist in the department_id column.

# Step 3.7 — Product Distribution by Department

## Objective

Analyze how products are distributed across departments in the product catalog.

In [18]:
### Data Preview — products table

con.execute("""SELECT * FROM products LIMIT 5""").df()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [27]:
con.execute("""SELECT d.department, COUNT(p.product_id) as Total_Products
                FROM products p
                LEFT JOIN departments d
                ON p.department_id = d.department_id
                GROUP BY d.department
                ORDER BY Total_Products DESC
                """).df()

,department,Total_Products
0,personal care,6563
1,snacks,6264
2,pantry,5371
3,beverages,4365
4,frozen,4007
5,dairy eggs,3449
6,household,3085
7,canned goods,2092
8,dry goods pasta,1858
9,produce,1684


## Observation

Products are unevenly distributed across departments.

The largest departments are Personal Care (6,563 products), Snacks (6,264 products), and Pantry (5,371 products).

The analysis shows that a significant portion of the product catalog is concentrated within a few major departments, while other departments contain substantially fewer products.

# Step 3.8 — Product Distribution by Aisle

## Objective

Analyze how products are distributed across aisles in the product catalog.

In [29]:
con.execute("""SELECT a.aisle, COUNT(p.product_id) AS Total_Products
                FROM aisles a
                LEFT JOIN products p
                ON a.aisle_id = p.aisle_id
                GROUP BY a.aisle
                ORDER BY Total_Products DESC """).df()

,aisle,Total_Products
0,missing,1258
1,candy chocolate,1246
2,ice cream ice,1091
3,vitamins supplements,1038
4,yogurt,1026
...,...,...
129,frozen juice,47
130,baby accessories,44
131,packaged produce,32
132,bulk grains rice dried goods,26


## Observation

Products are unevenly distributed across aisles.

The largest aisles are Missing (1,258 products), Candy Chocolate (1,246 products), and Ice Cream Ice (1,091 products).

The smallest aisles are Bulk Dried Fruits Vegetables (12 products), Bulk Grains Rice Dried Goods (26 products), and Packaged Produce (32 products).

This indicates that product concentration varies significantly across aisles, with some aisles containing a large portion of the catalog while others contain only a small number of products.

Note: The aisle named "missing" is a valid category in the dataset and does not represent SQL NULL values. Products assigned to this aisle belong to an uncategorized or placeholder aisle group.

# Step 3.9 — Largest Department

## Objective

Identify the department that contains the highest number of products.

In [32]:
con.execute("""SELECT d.department, COUNT(p.product_id) as Total_Products
                FROM departments d
                LEFT JOIN products p
                ON d.department_id = p.department_id
                GROUP BY d.department
                ORDER BY Total_Products DESC
                LIMIT 1""").df()
                

,department,Total_Products
0,personal care,6563


## Observation

The Personal Care department contains the highest number of products in the catalog with 6,563 products.

This indicates that Personal Care is the largest department in the Instacart product catalog based on product count.

# Step 3.10 — Largest Aisle

## Objective

Identify the aisle that contains the highest number of products.

In [34]:
con.execute("""SELECT a.aisle, COUNT(p.product_id) as Total_Products
                FROM aisles a
                LEFT JOIN products p
                ON a.aisle_id = p.aisle_id
                GROUP BY a.aisle
                ORDER BY COUNT(p.product_id) DESC
                LIMIT 1""").df()

,aisle,Total_Products
0,missing,1258


## Observation

The aisle named "missing" contains the highest number of products in the catalog with 1,258 products.

This indicates that the largest aisle in the Instacart product catalog is the "missing" aisle, which serves as a placeholder category for products that are not assigned to a more specific aisle.

# Step 4 — Product Catalog Summary

## Objective

Create a final summary of the product catalog by reviewing the key findings from the exploratory analysis.

Task:

Create a single query that returns:

Total Products,
Unique Products,
Total Departments,
Total Aisles,
Average Products Per Department

in one result table.

In [35]:
### Data Preview — products table

con.execute("""SELECT * FROM products LIMIT 5""").df()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [36]:
### Data Preview — departments table

con.execute("""SELECT * FROM departments LIMIT 5""").df()

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


In [37]:
### Data Preview — aisles table

con.execute("""SELECT * FROM aisles LIMIT 5""").df()

,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


In [46]:
con.execute("""SELECT COUNT(p.product_id) as Total_Products, COUNT(DISTINCT p.product_id) AS Unique_Products,
                COUNT(DISTINCT d.department_id) AS Total_Departments, COUNT(DISTINCT a.aisle_id) AS Total_Aisles,
                COUNT(p.product_id) / COUNT(DISTINCT d.department_id) as Average_Products_Per_Department

                FROM products p
                LEFT JOIN departments d
                ON p.department_id = d.department_id
                LEFT JOIN aisles a
                ON p.aisle_id = a.aisle_id""").df()

,Total_Products,Unique_Products,Total_Departments,Total_Aisles,Average_Products_Per_Department
0,49688,49688,21,134,2366.095238


## Observation

The products table contains 49,688 products and all product IDs are unique, confirming that no duplicate products exist in the catalog. The dataset is organized into 21 departments and 134 aisles, providing a structured hierarchy for product classification. On average, each department contains approximately 2,366 products, although the distribution is not uniform across departments. Overall, the product catalog passed all uniqueness, completeness, and relationship validation checks successfully.

# Step 5.1 — Above Average Aisles

## Objective

Identify aisles that contain a higher-than-average number of products and evaluate how product distribution varies across the catalog.

In [47]:
### Data Preview — products table

con.execute("""SELECT * FROM products LIMIT 5""").df()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [48]:
### Data Preview — aisles table

con.execute("""SELECT * FROM aisles LIMIT 5""").df()

,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


In [69]:
con.execute("""WITH Aisle_Count AS (
                                    SELECT a.aisle as aisle, COUNT(p.product_id) AS Total_Products
                                    FROM aisles a
                                    LEFT JOIN products p
                                    ON a.aisle_id = p.aisle_id
                                    GROUP BY a.aisle
                                    
                                    )

        SELECT *
        FROM Aisle_Count
        WHERE Total_Products > (SELECT AVG(Total_Products) AS AVG_Total_Products
                                FROM Aisle_Count
                                                )

                
                """).df()

,aisle,Total_Products
0,cookies cakes,874
1,dog food care,473
2,packaged cheese,891
3,tea,894
4,juice nectars,792
5,fresh vegetables,569
6,ice cream ice,1091
7,body lotions soap,504
8,hair care,816
9,energy granola bars,832


## Observation

The analysis used a Common Table Expression (CTE) to calculate the number of products contained within each aisle and then compared those counts against the overall average aisle size. The results show that several aisles contain substantially more products than the average aisle, indicating that the Instacart product catalog is not evenly distributed across categories. Product assortment is concentrated within a relatively small number of high-volume aisles, while many other aisles contain considerably fewer products. This highlights the presence of dominant product categories that contribute a disproportionate share of the overall catalog.

# Step 5.2 — Below Average Aisles

## Objective

Identify aisles that contain fewer products than the average aisle size and evaluate which categories have relatively small product assortments.

In [72]:
con.execute("""WITH Aisle_Count AS 

                (
                    SELECT a.aisle, COUNT(p.product_id) AS Total_Products
                    FROM aisles a
                    LEFT JOIN products p
                    ON a.aisle_id = p.aisle_id
                    GROUP BY a.aisle
                    )
                    
        SELECT *
        FROM Aisle_Count
        WHERE Total_Products < (SELECT AVG(Total_Products) as AVG_Total_Products
                                FROM Aisle_Count
                                 )

                    
                    """).df()

,aisle,Total_Products
0,grains rice dried goods,336
1,prepared meals,317
2,cocoa drink mixes,223
3,breakfast bars pastries,173
4,diapers wipes,187
...,...,...
77,fresh pasta,123
78,skin care,245
79,red wines,232
80,buns rolls,195


## Observation

The analysis identified a large number of aisles whose product counts fall below the average aisle size across the Instacart catalog. Categories such as Grains Rice Dried Goods, Prepared Meals, Cocoa Drink Mixes, Breakfast Bars Pastries, and Diapers Wipes contain significantly fewer products than the catalog's larger aisles. These results reinforce that product assortment is not evenly distributed across categories and follows a long-tail pattern, where a relatively small number of aisles contain a substantial portion of products while the majority of aisles maintain smaller inventories. This distribution suggests that product variety is concentrated within specific high-volume categories, whereas many aisles serve more specialized or niche product segments.

# Step 5.3 — Department Contribution Analysis

## Objective

Measure how much each department contributes to the overall product catalog and identify the departments that represent the largest share of products.

Create a query that returns:

Department Name,
Total Products,
Percentage Contribution to the Entire Product Catalog

The result should show how much of the catalog each department represents rather than only displaying raw product counts.

In [5]:
con.execute("""WITH Department_Distribution AS

                                  ( SELECT d.department, COUNT(p.product_id) AS Total_Products
                                    FROM products p
                                    LEFT JOIN departments d
                                    ON d.department_id = p.department_id
                                    GROUP BY d.department
                                    ORDER BY Total_Products DESC

                                                )

             SELECT *, ROUND(Total_Products * 100 / (SELECT COUNT(product_id) FROM products),2) AS Percentage_Contribution 
             FROM Department_Distribution
             
                
                
                
                """).df()

,department,Total_Products,Percentage_Contribution
0,personal care,6563,13.21
1,snacks,6264,12.61
2,pantry,5371,10.81
3,beverages,4365,8.78
4,frozen,4007,8.06
5,dairy eggs,3449,6.94
6,household,3085,6.21
7,canned goods,2092,4.21
8,dry goods pasta,1858,3.74
9,produce,1684,3.39


## Observation

The analysis shows that product distribution across departments is highly concentrated. Personal Care represents the largest share of the catalog, contributing approximately 13.2% of all products, followed closely by Snacks and Pantry. Together, the leading departments account for a substantial portion of the entire product catalog, indicating that Instacart allocates significant product variety to high-demand consumer categories. The percentage contribution metric provides a clearer view of catalog composition than raw product counts alone, making it easier to compare the relative importance of departments within the overall assortment.

# Step 5.4 — Department Share of Catalog

## Objective

Determine which departments individually contribute more than the average department contribution to the overall product catalog.

Task

Create a query that returns:

Department Name,
Total Products,
Percentage Contribution

Only return departments whose percentage contribution is above the average percentage contribution across all departments.

In [17]:
### Data Preview — departments table

con.execute("""SELECT * FROM departments LIMIT 5""").df()

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


In [18]:
### Data Preview — products table

con.execute("""SELECT * FROM products LIMIT 5""").df()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [35]:
con.execute(""" WITH department_distribution AS 

                            (
                            
                            SELECT d.department, COUNT(p.product_id) as Total_Products
                            FROM products p
                            LEFT JOIN departments d
                            ON d.department_id = p.department_id
                            GROUP BY d.department
                            ORDER BY Total_Products DESC
            
                            ),
                
                    percentage_distribution AS

                        ( 

                        SELECT * , ROUND(Total_Products * 100 / (SELECT COUNT(*) 
                                                          FROM products),2) AS Percentage_Contribution
                        FROM department_distribution

                          )

                    SELECT * 
                    FROM percentage_distribution
                
                    WHERE Percentage_Contribution > (SELECT AVG(Percentage_Contribution) AS Avg_Percentage_Contribution
                                                         FROM percentage_distribution)
                    
                
                
                """).df()

,department,Total_Products,Percentage_Contribution
0,personal care,6563,13.21
1,snacks,6264,12.61
2,pantry,5371,10.81
3,beverages,4365,8.78
4,frozen,4007,8.06
5,dairy eggs,3449,6.94
6,household,3085,6.21


## Observation

The analysis revealed that only a few departments contribute more than the average share of products in the catalog. Personal Care, Snacks, Pantry, Beverages, and Frozen collectively account for a substantial portion of all products available. This indicates that the product catalog is not evenly distributed across departments and is instead concentrated within a small number of high-volume categories. By combining CTEs with a subquery, we first calculated department-level contributions and then compared each department against the average contribution across all departments, allowing us to identify the most influential categories within the catalog.

# Step 5.5 — Department Contribution Ranking

## Objective

Identify how far each department is from the highest-contributing department in the product catalog.

## Task

Create a query that returns:

- Department
- Total_Products
- Percentage_Contribution
- Contribution_Gap

Where:

Contribution_Gap = Highest Department Percentage Contribution - Current Department Percentage Contribution

In [42]:
con.execute(""" WITH Department_Distribution AS 

                            (
                            SELECT d.department, COUNT(p.product_id) AS Total_Products
                            FROM products p
                            LEFT JOIN departments d
                            ON d.department_id = p.department_id
                            GROUP BY d.department
                            ORDER BY Total_Products DESC
                            ),

            Percentage_Contribution AS 
                            
                            (
                            SELECT * , ROUND(Total_Products * 100 / (SELECT COUNT(product_id)
                                                FROM products),2) AS Current_Contribution
                            FROM Department_Distribution
                            
                            )

                          SELECT * , ( (SELECT MAX(Current_Contribution) AS Max_Contribution
                                    FROM Percentage_Contribution) - Current_Contribution) AS Contributon_GAP
                        FROM Percentage_Contribution
                       
                
                
                
                """).df()

,department,Total_Products,Current_Contribution,Contributon_GAP
0,personal care,6563,13.21,0.00
1,snacks,6264,12.61,0.60
2,pantry,5371,10.81,2.40
3,beverages,4365,8.78,4.43
4,frozen,4007,8.06,5.15
5,dairy eggs,3449,6.94,6.27
6,household,3085,6.21,7.00
7,canned goods,2092,4.21,9.00
8,dry goods pasta,1858,3.74,9.47
9,produce,1684,3.39,9.82


## Observation

The analysis compared every department against the highest-contributing department in the catalog and quantified the difference using a contribution gap metric. Personal Care emerged as the leading department and therefore served as the benchmark with a contribution gap of zero. Departments such as Snacks and Pantry showed relatively small gaps, indicating that they are also major contributors to the overall catalog. This approach demonstrates how business analysts can measure the relative performance of categories against a top performer and quickly identify which departments are closest to the benchmark.

# Step 5.6 — Above Average Departments

## Objective

Identify departments whose product counts are above the average department size.

## Task

Create a query that returns:

- Department
- Total_Products

Only include departments whose product count is greater than the average department product count.

In [56]:
con.execute(""" With Department_Distribution AS
            
                            (
            
                            SELECT d.department, COUNT(p.product_id) as Total_Products
                            FROM products p
                            LEFT JOIN departments d
                            ON d.department_id = p.department_id
                            GROUP BY d.department
                            

                            ),

                    AVG_department_Size AS

                            (
                            
                            SELECT * , (SELECT AVG(Total_Products)
                                        FROM Department_Distribution) AS AVG_department_Products
                                        
                            FROM Department_Distribution

                            )

                            SELECT * 
                            FROM AVG_department_Size
                            WHERE Total_Products > AVG_department_Products
                            ORDER BY Total_Products DESC

                            """).df()


                

,department,Total_Products,AVG_department_Products
0,personal care,6563,2366.095238
1,snacks,6264,2366.095238
2,pantry,5371,2366.095238
3,beverages,4365,2366.095238
4,frozen,4007,2366.095238
5,dairy eggs,3449,2366.095238
6,household,3085,2366.095238


## Observation

The analysis identified departments whose product counts exceed the average department size across the catalog. By first calculating department-level product counts and then establishing an average benchmark, we were able to isolate the largest departments within the catalog. The results show that only a handful of departments contain substantially more products than the average department, highlighting an uneven distribution of products across categories. This exercise demonstrated how multiple CTEs can be used to separate metric generation from benchmark calculation, making complex analytical queries easier to understand and maintain.

# Step 6.1 — Department Size Classification

## Objective

Classify departments into size categories based on their product counts using conditional logic.

## Task

Create a query that returns:

- Department
- Total_Products
- Department_Size

Classify departments using the following rules:

Large Department
→ More than 5,000 products

Medium Department
→ Between 2,000 and 5,000 products

Small Department
→ Less than 2,000 products

In [67]:
con.execute("""WITH Total_Products AS

                (   SELECT d.department, COUNT(p.product_id) AS Total_Products,
                    FROM products p
                    LEFT JOIN departments d
                    ON d.department_id = p.department_id
                    GROUP BY d.department
                    )

        SELECT *, CASE 
                    WHEN Total_Products > 5000 THEN 'Large_Department'
                    WHEN Total_Products BETWEEN 2000 AND 5000 THEN 'Medium_Department'
                    WHEN Total_Products < 2000 THEN 'Small_Department' END AS Department_Size

                FROM Total_Products
                ORDER BY Total_Products DESC

                    
                    
            """).df()




,department,Total_Products,Department_Size
0,personal care,6563,Large_Department
1,snacks,6264,Large_Department
2,pantry,5371,Large_Department
3,beverages,4365,Medium_Department
4,frozen,4007,Medium_Department
5,dairy eggs,3449,Medium_Department
6,household,3085,Medium_Department
7,canned goods,2092,Medium_Department
8,dry goods pasta,1858,Small_Department
9,produce,1684,Small_Department


## Observation

Departments were classified into Large, Medium, and Small categories using a CASE WHEN expression based on their product counts. This transformed a numeric metric into a business-friendly category, making the results easier to interpret. The analysis showed that only a few departments fall into the Large category, while most departments belong to the Medium or Small groups.

# Step 6.2 — Product Popularity Buckets

## Objective

Classify products into popularity groups based on the number of times they were ordered.

## Task

Return:

- product_id
- product_name
- Total_Orders
- Popularity_Category

Use the following rules:

Highly Popular
→ More than 1,000 orders

Moderately Popular
→ Between 100 and 1,000 orders

Low Popularity
→ Less than 100 orders

In [68]:
### Data Preview — products table

con.execute("""SELECT * FROM products LIMIT 5""").df()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [71]:
### Data Preview — order_products table

con.execute("""SELECT * FROM order_products LIMIT 5""").df()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [73]:
con.execute("""WITH Total_Orders_CTE AS 

                (
                    SELECT op.product_id, p.product_name, COUNT(op.order_id) as Total_Orders
                    FROM order_products op
                    LEFT JOIN products p
                    ON op.product_id = p.product_id
                    GROUP BY op.product_id, p.product_name
                    
                    )

                SELECT * , CASE
                                WHEN Total_Orders < 100 THEN 'Low_Popularity'
                                WHEN Total_Orders BETWEEN 100 AND 1000 THEN 'Moderately_Popular'
                                ELSE 'Highly Popular' END AS Popularity_Category
                    
                    FROM Total_Orders_CTE
                    ORDER BY Total_Orders DESC
                    
                    
                    """).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,product_id,product_name,Total_Orders,Popularity_Category
0,24852,Banana,472565,Highly Popular
1,13176,Bag of Organic Bananas,379450,Highly Popular
2,21137,Organic Strawberries,264683,Highly Popular
3,21903,Organic Baby Spinach,241921,Highly Popular
4,47209,Organic Hass Avocado,213584,Highly Popular
...,...,...,...,...
49672,1002,All Natural Stevia Liquid Extract Sweetener,1,Low_Popularity
49673,15901,Bite Size Caramel Chocolates,1,Low_Popularity
49674,8435,Vitamin D Gummies,1,Low_Popularity
49675,32126,Kefir Raspberry,1,Low_Popularity


## Observation

Products were classified into popularity categories based on the number of times they appeared in customer orders. By aggregating order counts at the product level and applying a CASE WHEN expression, products were grouped into Low Popularity, Moderately Popular, and Highly Popular segments. The results revealed that a small number of products, such as bananas and other staple grocery items, were ordered significantly more often than most products in the catalog.

# Step 6.3 — Reorder Behavior Classification

## Objective

Classify products based on customer reorder behavior.

## Task

Return:

- product_id
- product_name
- Total_Orders
- Total_Reorders
- Reorder_Rate
- Reorder_Category

Use the following rules:

Frequently Reordered
→ Reorder Rate greater than 70%

Occasionally Reordered
→ Reorder Rate between 30% and 70%

Rarely Reordered
→ Reorder Rate less than 30%

In [84]:
con.execute("""WITH Products_Count AS

                (
                SELECT op.product_id, p.product_name, COUNT(op.order_id) AS Total_Orders,
                SUM(op.reordered) as Total_Reorders
                FROM order_products op
                LEFT JOIN products p
                ON op.product_id = p.product_id
                GROUP BY op.product_id, p.product_name
                
            ),

            Reordered_Rate AS

            ( 
                SELECT * , ROUND(Total_Reorders * 100 / Total_Orders,2) AS Reorder_Rate
                FROM Products_Count

                )

            SELECT * , CASE
                            WHEN Reorder_Rate < 30 THEN 'Rarely_Reordered'
                            WHEN Reorder_Rate BETWEEN 30 AND 70 THEN 'Occasionally_Reordered'
                            ELSE 'Frequently_Reordered' END AS Reorder_Category
                            
            FROM Reordered_Rate
            ORDER BY Reorder_Category
                
                
                """).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,product_id,product_name,Total_Orders,Total_Reorders,Reorder_Rate,Reorder_Category
0,4952,Unsweet Peach Water,1781,1298.0,72.88,Frequently_Reordered
1,32589,Tender Care Diapers Jumbo Pack - Size 3,47,35.0,74.47,Frequently_Reordered
2,1559,Cherry Pomegranate Greek Yogurt,6858,5023.0,73.24,Frequently_Reordered
3,1743,Super Scoop Clumping Fragrance Free Cat Litter,185,131.0,70.81,Frequently_Reordered
4,49519,Raspberry Essence Water,2029,1510.0,74.42,Frequently_Reordered
...,...,...,...,...,...,...
49672,27152,"Jelly Beans, Sours",9,0.0,0.00,Rarely_Reordered
49673,44134,"Olive Spray Oil, Extra Virgin, Non-Aerosol",33,9.0,27.27,Rarely_Reordered
49674,27812,"Adult Grain-Free Chicken Thigh, Carrots & Swee...",4,0.0,0.00,Rarely_Reordered
49675,3984,Cherry Bites,13,2.0,15.38,Rarely_Reordered


# Step 6.4 — Conditional Aggregation

## Objective

Analyze reorder behavior at the product level using conditional aggregation.

## Task

Return:

- product_id
- product_name
- Total_Orders
- Reordered_Orders
- First_Time_Orders

Where:

- Reordered_Orders represents purchases where reordered = 1
- First_Time_Orders represents purchases where reordered = 0

In [85]:
con.execute("""SELECT op.product_id, p.product_name, COUNT(op.order_id) AS Total_Orders,
                SUM(CASE
                        WHEN reordered = 1 THEN 1 ELSE 0 END) AS Reordered_Orders,
                SUM(CASE 
                        WHEN reordered = 0 THEN 1 ELSE 0 END) AS First_Time_Orders
                        
                FROM order_products op
                LEFT JOIN products p
                ON op.product_id = p.product_id
                GROUP BY op.product_id, p.product_name
                ORDER BY Total_Orders DESC """).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,product_id,product_name,Total_Orders,Reordered_Orders,First_Time_Orders
0,24852,Banana,472565,398609.0,73956.0
1,13176,Bag of Organic Bananas,379450,315913.0,63537.0
2,21137,Organic Strawberries,264683,205845.0,58838.0
3,21903,Organic Baby Spinach,241921,186884.0,55037.0
4,47209,Organic Hass Avocado,213584,170131.0,43453.0
...,...,...,...,...,...
49672,10584,Homestlye Cornbread Stuffing,1,0.0,1.0
49673,24402,Orangemint Flavored Water,1,0.0,1.0
49674,32464,Flavor Snacks,1,0.0,1.0
49675,3426,Organic Better Rest Tea Blend,1,0.0,1.0


## Observation

The highest-volume products generated substantially more reordered purchases than first-time purchases, indicating strong repeat buying behavior. Top-selling products such as Banana and Bag of Organic Bananas were reordered by customers far more frequently than they were purchased for the first time. Products with very low order volumes consisted primarily of first-time purchases and showed little to no reordering activity.

# Step 6.5 — Reorder Rate Calculation

## Objective

Measure customer repeat-purchase behavior by calculating the reorder rate for each product.

## Task

Return:

- product_id
- product_name
- Total_Orders
- Reordered_Orders
- Reorder_Rate

Calculate the percentage of orders that were reordered for each product.

In [94]:
con.execute(""" WITH Total_Reordered AS

                (
                    SELECT op.product_id, p.product_name, COUNT(op.order_id) as Total_Orders,
                    SUM(CASE
                            WHEN op.reordered = 1 THEN 1 ELSE 0 END) AS Reordered_Orders
                    FROM order_products op
                    LEFT JOIN products p
                    ON op.product_id = p.product_id
                    GROUP BY op.product_id, p.product_name
                    HAVING Total_Orders >= 100

                    )

                SELECT * , ROUND(Reordered_Orders * 100 / Total_Orders,2) AS Reorder_Rate
                FROM Total_Reordered
                ORDER BY Reorder_Rate DESC
                    
                    
                    
                    """).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,product_id,product_name,Total_Orders,Reordered_Orders,Reorder_Rate
0,27740,Chocolate Love Bar,101,93.0,92.08
1,35604,Maca Buttercups,100,90.0,90.00
2,38251,Benchbreak Chardonnay,111,99.0,89.19
3,10236,Fragrance Free Clay with Natural Odor Eliminat...,129,113.0,87.60
4,20598,Thousand Island Salad Snax,112,98.0,87.50
...,...,...,...,...,...
20062,14294,"Cornmeal, Stone Ground",123,2.0,1.63
20063,13373,Food Coloring,123,2.0,1.63
20064,24364,Organic Chinese Five Spice,169,2.0,1.18
20065,10376,Organic Caraway Seeds,134,1.0,0.75


# Step 6.6 — Conditional Aggregation by Order Size

## Objective

Analyze customer ordering behavior by separating products purchased in small orders versus large orders.

## Task

Return:

- product_id
- product_name
- Total_Orders
- Small_Order_Purchases
- Large_Order_Purchases

Treat:

- Small Order = order contains 10 or fewer products
- Large Order = order contains more than 10 products

In [101]:
### Data Preview — products table

con.execute("""SELECT * FROM products LIMIT 5""").df()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [100]:
### Data Preview — order_products table

con.execute("""SELECT * FROM order_products LIMIT 5""").df()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [111]:
con.execute("""WITH Order_Size AS

            (
                SELECT order_id, COUNT(*) AS Total_Products
                FROM order_products
                GROUP BY order_id

            ),

           Total_Products AS

           (
           
           SELECT op.product_id, os.Total_Products
            FROM Order_Size os
            LEFT JOIN order_products op
            ON op.order_id = os.order_id
           

            )

            SELECT tp.product_id, p.product_name, COUNT(*) AS Total_Orders,
            SUM(
                CASE
                    WHEN Total_Products <= 10 THEN 1 ELSE 0 END) AS Small_Order_Purchases,
            SUM(
                CASE
                    WHEN Total_Products > 10 THEN 1 ELSE 0 END) AS Large_Order_Purchases
            
            FROM Total_Products tp
            LEFT JOIN products p
            ON p.product_id = tp.product_id
            GROUP BY tp.product_id, p.product_name

            
                        
                        """).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,product_id,product_name,Total_Orders,Small_Order_Purchases,Large_Order_Purchases
0,14678,Organic Frozen Peas,20228,5474.0,14754.0
1,47367,Grape Flavored Pain Reliever and Fever Reducer,168,85.0,83.0
2,25154,Chicken Broccoli Supreme,434,152.0,282.0
3,13874,Organic Pumpkin,4594,1484.0,3110.0
4,41950,Organic Tomato Cluster,64289,19288.0,45001.0
...,...,...,...,...,...
49672,1002,All Natural Stevia Liquid Extract Sweetener,1,1.0,0.0
49673,25719,"Iced Tea, Lemon",2,1.0,1.0
49674,41923,Half & Half Peach Lemonade Tea,3,2.0,1.0
49675,33213,"Chili with Pinto Beans, Spelt and Spices",4,0.0,4.0


## Observation

Products exhibited different purchase patterns across order sizes. Some products appeared predominantly in larger orders, while others showed a more balanced distribution between small and large orders. For each product, Small_Order_Purchases and Large_Order_Purchases collectively accounted for Total_Orders.

# Step 6.7 — Order Composition Analysis

## Objective

Analyze whether products are more commonly purchased in single-item orders or multi-item basket orders.

## Task

Return:

- product_id
- product_name
- Total_Orders
- Single_Item_Order_Purchases
- Multi_Item_Order_Purchases

In [112]:
### Data Preview — order_products table

con.execute("""SELECT * FROM order_products LIMIT 5""").df()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [113]:
### Data Preview — products table

con.execute("""SELECT * FROM products LIMIT 5""").df()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [125]:
con.execute(""" WITH Order_Size AS

                    (
                        SELECT order_id, COUNT(*) AS Total_Products
                        FROM order_products
                        GROUP BY order_id
            
                        ),

             Total_Products_Size AS 

                    (
                        SELECT op.product_id, os.Total_Products
                        FROM Order_Size os
                        LEFT JOIN order_products op
                        ON op.order_id = os.order_id

                    )

                    SELECT tp.product_id, p.product_name, COUNT(*) as Total_Orders,
                    SUM(CASE
                            WHEN Total_Products = 1 THEN 1 ELSE 0 END) AS Single_Item_Order_Purchases,
                    SUM(CASE
                            WHEN Total_Products > 1 THEN 1 ELSE 0 END) AS Multi_Item_Order_Purchases

                    FROM Total_Products_Size tp
                    LEFT JOIN products p
                    ON tp.product_id = p.product_id
                    GROUP BY tp.product_id, p.product_name
                    ORDER BY Total_Orders DESC
            
            
            """).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,product_id,product_name,Total_Orders,Single_Item_Order_Purchases,Multi_Item_Order_Purchases
0,24852,Banana,472565,2047.0,470518.0
1,13176,Bag of Organic Bananas,379450,2774.0,376676.0
2,21137,Organic Strawberries,264683,1120.0,263563.0
3,21903,Organic Baby Spinach,241921,1166.0,240755.0
4,47209,Organic Hass Avocado,213584,697.0,212887.0
...,...,...,...,...,...
49672,38977,Original Jerky,1,0.0,1.0
49673,16461,Peas Pulav Basmati Rice With Green Peas,1,0.0,1.0
49674,48343,Hennepin Farmhouse Ale,1,0.0,1.0
49675,22747,Vanilla Bean Sheep Milk Ice Cream,1,0.0,1.0


## Observation

Most high-volume products were purchased predominantly within multi-item baskets rather than single-item orders. Single-item purchases represented a relatively small portion of total product purchases, indicating that customers typically purchased these products alongside other items within larger shopping baskets.

# Step 7.1 — Product Purchase Ranking

## Objective

Rank products based on their total purchase frequency across all orders.

## Task

Generate a purchase ranking for all products using a window function.

Return the following columns:

- product_id
- product_name
- total_purchases
- purchase_rank

In [6]:
### Data Preview — products table

con.execute("""SELECT * FROM products LIMIT 5""").df()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [7]:
### Data Preview — order_products table

con.execute("""SELECT * FROM order_products LIMIT 5""").df()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [10]:
con.execute("""SELECT p.product_id, p.product_name, COUNT(op.order_id) as Total_Purchases,
                      ROW_NUMBER() OVER(ORDER BY COUNT(op.order_id) DESC) as Purchase_Rank
                FROM products p
                LEFT JOIN order_products op
                ON op.product_id = p.product_id
                GROUP BY p.product_id, p.product_name""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,product_id,product_name,Total_Purchases,Purchase_Rank
0,24852,Banana,472565,1
1,13176,Bag of Organic Bananas,379450,2
2,21137,Organic Strawberries,264683,3
3,21903,Organic Baby Spinach,241921,4
4,47209,Organic Hass Avocado,213584,5
...,...,...,...,...
49683,43725,Sweetart Jelly Beans,0,49684
49684,37703,Ultra Sun Blossom Liquid 90 loads Fabric Enhan...,0,49685
49685,49540,Pure Squeezed Lemonade,0,49686
49686,3630,Protein Granola Apple Crisp,0,49687


## Observation

Product purchase frequency varied considerably across the catalog, with a small number of products occupying the highest positions in the ranking. The ranking established a clear ordering of products based on purchase volume, providing an overall view of product popularity. Each product was assigned a unique position, producing a complete sequential ranking across the dataset.

# Step 7.2 — Product Reorder Ranking

## Objective

Rank products based on the total number of reorders.

## Task

Generate a ranking of products by their total reorder count using a window function.

Return the following columns:

- product_id
- product_name
- total_reorders
- reorder_rank

In [14]:
con.execute("""SELECT p.product_id, p.product_name, COUNT(op.reordered) as Total_Reorders,
                      ROW_NUMBER() OVER(ORDER BY Total_Reorders DESC) as Reorder_Rank
                FROM products p
                LEFT JOIN order_products op
                ON op.product_id = p.product_id
                GROUP BY p.product_id, p.product_name""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,product_id,product_name,Total_Reorders,Reorder_Rank
0,24852,Banana,472565,1
1,13176,Bag of Organic Bananas,379450,2
2,21137,Organic Strawberries,264683,3
3,21903,Organic Baby Spinach,241921,4
4,47209,Organic Hass Avocado,213584,5
...,...,...,...,...
49683,7045,Unpeeled Apricot Halves in Heavy Syrup,0,49684
49684,45971,12 Inch Taper Candle White,0,49685
49685,36233,Water With Electrolytes,0,49686
49686,37703,Ultra Sun Blossom Liquid 90 loads Fabric Enhan...,0,49687


## Observation

Reorder activity was concentrated among a relatively small set of products, while many products recorded little or no reorder activity. The ranking highlighted products with the highest customer repeat purchases, revealing substantial variation in reorder frequency across the catalog.

# Step 7.3 — Product Purchase Ranking with RANK()

## Objective

Rank products by their total purchase frequency while allowing products with equal purchase counts to share the same position.

## Task

Generate a purchase ranking using `RANK()`.

Return the following columns:

- product_id
- product_name
- total_purchases
- purchase_rank

In [7]:
con.execute("""SELECT p.product_id, p.product_name, COUNT(op.order_id) AS total_purchases,
                      RANK() OVER(Order by total_purchases DESC) as purchase_rank
                FROM products p
                LEFT JOIN order_products op
                ON p.product_id = op.product_id
                GROUP BY p.product_id, p.product_name""").df()

,product_id,product_name,total_purchases,purchase_rank
0,24852,Banana,472565,1
1,13176,Bag of Organic Bananas,379450,2
2,21137,Organic Strawberries,264683,3
3,21903,Organic Baby Spinach,241921,4
4,47209,Organic Hass Avocado,213584,5
...,...,...,...,...
49683,3630,Protein Granola Apple Crisp,0,49678
49684,25383,Chocolate Go Bites,0,49678
49685,27499,Non-Dairy Coconut Seven Layer Bar,0,49678
49686,7045,Unpeeled Apricot Halves in Heavy Syrup,0,49678


## Observation

Multiple products shared identical purchase frequencies, resulting in tied ranking positions. Products with equal purchase counts received the same rank, producing repeated ranking values within the overall product ranking. The ranking highlighted the distribution of purchase activity while preserving ties among products with identical purchase volumes.

# Step 7.4 — Product Purchase Ranking with DENSE_RANK()

## Objective

Rank products by their total purchase frequency while assigning consecutive ranking positions to products with equal purchase counts.

## Task

Generate a purchase ranking using `DENSE_RANK()`.

Return the following columns:

- product_id
- product_name
- total_purchases
- purchase_rank

In [9]:
con.execute(""" SELECT p.product_id, p.product_name, COUNT(op.order_id) as total_purchases,
                        DENSE_RANK() OVER(ORDER BY total_purchases DESC) as purchase_rank
                FROM products p
                LEFT JOIN order_products op
                ON p.product_id = op.product_id
                GROUP BY p.product_id, p.product_name""").df()

,product_id,product_name,total_purchases,purchase_rank
0,24852,Banana,472565,1
1,13176,Bag of Organic Bananas,379450,2
2,21137,Organic Strawberries,264683,3
3,21903,Organic Baby Spinach,241921,4
4,47209,Organic Hass Avocado,213584,5
...,...,...,...,...
49683,37703,Ultra Sun Blossom Liquid 90 loads Fabric Enhan...,0,4162
49684,3630,Protein Granola Apple Crisp,0,4162
49685,25383,Chocolate Go Bites,0,4162
49686,43725,Sweetart Jelly Beans,0,4162


In [13]:
### Data Preview — products table

con.execute("""SELECT * FROM products LIMIT 5""").df()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [14]:
### Data Preview — order_products table

con.execute("""SELECT * FROM order_products LIMIT 5""").df()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [15]:
### Data Preview — department table

con.execute("""SELECT * FROM departments LIMIT 5""").df()

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


# Step 7.5 — Top Reordered Product Per Department

## Objective

Identify the highest reordered product within each department.

## Task

For each department, rank products by their total reorders and return only the highest ranked product.

Return the following columns:

- department_id
- department
- product_id
- product_name
- total_reorders
- department_rank

In [35]:
con.execute("""WITH product_Reorders AS 
                 (
                      SELECT p.department_id, p.product_id, p.product_name, SUM(op.reordered) as Total_Reorders
                      FROM products p
                      LEFT JOIN order_products op
                      ON p.product_id = op.product_id
                      GROUP BY p.department_id, p.product_id, p.product_name

                      ),

                    Ranked_Departments AS 

                (
                      SELECT pr.product_id, pr.product_name, pr.Total_Reorders, d.department, pr.department_id,
                      RANK() OVER(PARTITION BY pr.department_id ORDER BY pr.Total_Reorders DESC) as department_rank
                      FROM product_Reorders pr
                      LEFT JOIN departments d
                      ON pr.department_id = d.department_id
                      

                      )

                      SELECT *
                      FROM Ranked_Departments
                      WHERE department_rank = 1
                      
                      
                      """).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,product_id,product_name,Total_Reorders,department,department_id,department_rank
0,5161,Dried Mango,7583.0,bulk,10,1
1,27156,Organic Black Beans,22352.0,canned goods,15,1
2,25890,Boneless Skinless Chicken Breasts,33286.0,meat seafood,12,1
3,24852,Banana,398609.0,produce,4,1
4,38662,Roasted Almond Butter,2709.0,other,2,1
5,41844,Honey Nut Cheerios,15340.0,breakfast,14,1
6,5077,100% Whole Wheat Bread,44834.0,bakery,3,1
7,36724,Organic Sea Salt Roasted Seaweed Snacks,6580.0,international,6,1
8,9047,Premium Epsom Salt,2341.0,personal care,11,1
9,43875,Baby Food Stage 2 Blueberry Pear & Purple Carrot,5685.0,babies,18,1


## Observation

Reorder activity was distributed differently across departments, with each department exhibiting a distinct top-performing product. Department-level ranking isolated the highest reordered product while maintaining independent rankings across categories. The final result provided a concise view of the strongest repeat-purchase product within every department.

# Step 7.6 — Previous Order Comparison

## Objective

Compare each order with the customer's previous order.

## Task

For every customer, identify each order along with the number of days since their previous order.

Return the following columns:

- user_id
- order_id
- order_number
- days_since_prior_order
- previous_order_days

In [36]:
### Data Preview — order table

con.execute("""SELECT * FROM orders LIMIT 5""").df()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,08,NaN
1,2398795,1,prior,2,3,07,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,07,29.0
4,431534,1,prior,5,4,15,28.0


In [45]:
con.execute("""SELECT user_id, order_id, order_number, days_since_prior_order,
                LAG(days_since_prior_order) OVER(PARTITION BY user_id ORDER BY order_number) as previous_order_days
                
               FROM orders """).df()

,user_id,order_id,order_number,days_since_prior_order,previous_order_days
0,55,2080943,1,NaN,NaN
1,55,2006658,2,9.0,NaN
2,55,3057843,3,28.0,9.0
3,55,3114269,4,13.0,28.0
4,55,2986671,5,15.0,13.0
...,...,...,...,...,...
3421078,60128,2949868,9,4.0,13.0
3421079,60128,765990,10,30.0,4.0
3421080,60128,1869747,11,19.0,30.0
3421081,60128,434066,12,15.0,19.0


## Observation

The previous order interval was successfully aligned with each customer's order history, enabling sequential comparison across purchases. The first order for every customer contained no previous interval, while subsequent orders preserved the interval from the immediately preceding purchase. The result established a chronological view of customer purchasing behavior.

In [46]:
### Data Preview — order table

con.execute("""SELECT * FROM orders LIMIT 5""").df()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,08,NaN
1,2398795,1,prior,2,3,07,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,07,29.0
4,431534,1,prior,5,4,15,28.0


In [47]:
### Data Preview — order_products table

con.execute("""SELECT * FROM order_products LIMIT 5""").df()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


# Step 7.7 — Customer Reorder Behavior Comparison

## Objective

Compare the number of reordered products in each order with the customer's previous order.

## Task

For every customer, calculate the total reordered products in each order and compare it with the previous order.

Return the following columns:

- user_id
- order_id
- order_number
- total_reordered_products
- previous_reordered_products
- reorder_difference

In [50]:
con.execute("""WITH reordered_products AS 
                (
                
                   SELECT o.user_id, o.order_id, o.order_number, SUM(op.reordered) AS total_reordered_products,
                      LAG(total_reordered_products) OVER (Partition by user_id ORDER BY order_number) as previous_reordered_products
               FROM orders o
               LEFT JOIN order_products op
               ON o.order_id = op.order_id
               GROUP BY o.user_id, o.order_id, o.order_number
               
               )
               
               SELECT *, (total_reordered_products - previous_reordered_products) as reorder_difference
               FROM reordered_products
               
               """).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,user_id,order_id,order_number,total_reordered_products,previous_reordered_products,reorder_difference
0,223,967329,1,0.0,NaN,NaN
1,223,699950,2,3.0,0.0,3.0
2,223,2650039,3,8.0,3.0,5.0
3,223,138290,4,5.0,8.0,-3.0
4,223,302797,5,7.0,5.0,2.0
...,...,...,...,...,...,...
3421078,36884,2969395,9,10.0,8.0,2.0
3421079,36884,872582,10,10.0,10.0,0.0
3421080,36884,923218,11,8.0,10.0,-2.0
3421081,36884,618596,12,9.0,8.0,1.0


## Observation

Customer reorder activity fluctuated across consecutive orders, with both positive and negative changes observed throughout purchasing histories. Sequential comparison highlighted shifts in repeat purchasing behavior while preserving the chronological progression of each customer's orders. The derived difference metric quantified changes in reorder activity between consecutive purchases.

# Step 7.8 — Largest Increase in Basket Size

## Objective

Identify the largest increase in basket size between consecutive orders for each customer.

## Task

For every customer, calculate the basket size of each order, compare it with the previous order, and identify the order that experienced the largest increase in basket size.

Return the following columns:

- user_id
- order_id
- order_number
- basket_size
- previous_basket_size
- basket_size_increase

In [68]:
con.execute("""WITH basket_size AS

        (

            SELECT order_id, COUNT(*) as current_basket_size
            FROM order_products 
            GROUP BY order_id

            ),

            Previous_Basket AS

            (

            SELECT o.user_id, o.order_id, o.order_number, bs.current_basket_size, 
                   LAG(bs.current_basket_size) OVER(Partition by o.user_id ORDER BY o.order_number) AS previous_basket_size
            FROM orders o
            LEFT JOIN basket_size bs
            ON o.order_id = bs.order_id

            ),

            Ranked_Basket_Size AS

            (

            SELECT user_id, order_id, order_number, current_basket_size, previous_basket_size,
                    (current_basket_size - previous_basket_size) AS basket_size_increase, 
                    RANK() OVER(Partition by user_id ORDER BY basket_size_increase DESC) as Rank
            FROM Previous_Basket

            )

            SELECT *
            FROM Ranked_Basket_Size
            WHERE rank = 1
            
            


        """).df()

,user_id,order_id,order_number,current_basket_size,previous_basket_size,basket_size_increase,Rank
0,126,1879545,6,14,8,6,1
1,138,2226790,26,6,1,5,1
2,161,651160,6,22,7,15,1
3,197,843713,6,29,15,14,1
4,202,1643268,5,47,7,40,1
...,...,...,...,...,...,...,...
241132,206082,1882829,8,17,7,10,1
241133,206106,548165,6,7,4,3,1
241134,206115,2534112,4,25,6,19,1
241135,206153,1138542,6,16,4,12,1


## Observation

Basket sizes changed considerably across consecutive customer orders, with each customer exhibiting a distinct maximum increase during their purchasing history. Ranking basket size differences isolated the largest positive change while preserving the chronological relationship between consecutive orders. The final result identified the most significant basket growth event for every customer.

# Step 7.9 — Next Order Comparison

## Objective

Compare each order with the customer's next order.

## Task

For every customer, identify each order along with the number of days until their next order.

Return the following columns:

- user_id
- order_id
- order_number
- days_since_prior_order
- next_order_days

In [70]:
con.execute("""SELECT user_id, order_id, order_number, days_since_prior_order,
                LEAD(days_since_prior_order) OVER(Partition by user_id ORDER BY order_number) next_order_days
                FROM orders""").df()

,user_id,order_id,order_number,days_since_prior_order,next_order_days
0,7,2565571,1,NaN,30.0
1,7,2402008,2,30.0,30.0
2,7,121053,3,30.0,9.0
3,7,1695742,4,9.0,3.0
4,7,3321109,5,3.0,10.0
...,...,...,...,...,...
3421078,100993,926031,15,10.0,18.0
3421079,100993,218439,16,18.0,13.0
3421080,100993,959017,17,13.0,5.0
3421081,100993,3279237,18,5.0,10.0


## Observation

The next purchase interval was successfully aligned with each customer's order history, enabling forward-looking comparison across consecutive purchases. The final order for each customer contained no subsequent interval, while earlier orders preserved the interval to the immediately following purchase. The result established a chronological view of upcoming purchasing behavior.

# Step 7.10 — Next Order Basket Size Comparison

## Objective

Compare the basket size of each order with the customer's next order.

## Task

For every customer, calculate the basket size of each order and compare it with the basket size of their immediately following order.

Return the following columns:

- user_id
- order_id
- order_number
- current_basket_size
- next_basket_size
- basket_size_difference

In [75]:
con.execute("""WITH basket_size AS

            (

                SELECT order_id, COUNT(*) AS current_basket_size
                FROM order_products
                GROUP BY order_id

                ),

                next_basket_size AS

            (

                SELECT o.order_id, o.order_number, bs.current_basket_size,
                        LEAD(bs.current_basket_size) OVER(PARTITION BY o.user_id ORDER BY o.order_number) as next_basket_size
                FROM basket_size bs
                LEFT JOIN orders o
                ON o.order_id = bs.order_id

                )

                SELECT * , (next_basket_size - current_basket_size) as basket_size_difference
                FROM next_basket_size 

                """).df()

,order_id,order_number,current_basket_size,next_basket_size,basket_size_difference
0,2080943,1,10,9,-1
1,2006658,2,9,8,-1
2,3057843,3,8,24,16
3,3114269,4,24,15,-9
4,2986671,5,15,10,-5
...,...,...,...,...,...
3214869,2670581,9,7,7,0
3214870,2330177,10,7,<NA>,<NA>
3214871,2188726,1,11,20,9
3214872,2142915,2,20,14,-6


## Observation

Basket size changes varied considerably between consecutive orders, with both substantial increases and decreases observed across users. Most order transitions involved relatively small adjustments, while a smaller number of users exhibited sharp shifts in purchasing behavior. Tracking the next order alongside the current basket size highlighted how shopping patterns evolved over time rather than across isolated purchases.

# Step 7.8 — Lead Function: Next Basket Growth Ranking

## Objective

Identify the largest increase in basket size between consecutive orders for each user.

## Task

Return the user’s highest basket size increase from one order to the next.

- Output 1: Current and next basket sizes for consecutive orders
- Output 2: Basket size increase between consecutive orders
- Output 3: The largest basket size increase for each user
## Return the following columns exactly:

- order_id
- order_number
- current_basket_size
- next_basket_size
- basket_size_increase
- basket_growth_rank

In [82]:
con.execute("""WITH basket_size AS

            ( 
                SELECT order_id, COUNT(*) AS current_basket_size
                FROM order_products
                GROUP BY order_id

            ), 
            
            next_basket_size AS

            (

                SELECT o.user_id, bs.order_id, o.order_number, bs.current_basket_size,
                LEAD(bs.current_basket_size) OVER(PARTITION BY o.user_id ORDER BY o.order_number) as next_basket_size
                FROM basket_size bs
                LEFT JOIN orders o
                ON o.order_id = bs.order_id

                ), 

                basket_size_increase AS

                (

                SELECT * , (next_basket_size - current_basket_size) as basket_size_increase,
                      RANK() OVER(PARTITION BY user_id ORDER BY basket_size_increase DESC) as basket_growth_rank
                FROM next_basket_size

                )

                SELECT *
                FROM basket_size_increase
                where basket_growth_rank = 1

                """).df()
                

            

,user_id,order_id,order_number,current_basket_size,next_basket_size,basket_size_increase,basket_growth_rank
0,7,2565571,1,12,21,9,1
1,57,839442,11,2,7,5,1
2,77,1574973,4,8,18,10,1
3,80,1331103,1,2,13,11,1
4,84,1726905,6,10,16,6,1
...,...,...,...,...,...,...,...
241132,87651,2547618,10,4,18,14,1
241133,87675,2999800,10,5,27,22,1
241134,87676,305186,10,6,22,16,1
241135,87680,2349045,38,7,23,16,1


## Observation

The magnitude of basket growth varied substantially across customers, with each user exhibiting a distinct peak increase between consecutive orders. Ranking these changes highlighted the most significant shift in purchasing behavior for every customer while preserving the chronological sequence of orders. Large positive jumps were relatively uncommon compared to modest basket size fluctuations.

In [83]:
### Data Preview — order table

con.execute("""SELECT * FROM orders LIMIT 5""").df()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,08,NaN
1,2398795,1,prior,2,3,07,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,07,29.0
4,431534,1,prior,5,4,15,28.0


In [84]:
### Data Preview — order_products table

con.execute("""SELECT * FROM order_products LIMIT 5""").df()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


# Step 7.9 — Lead Function: Days Until Next Purchase Change

## Objective

Measure how the waiting time between purchases changes from one order to the next.

## Task

Return the following columns:

- user_id
- order_id
- order_number
- days_since_prior_order
- next_order_days
- days_difference

Where:

days_difference = next_order_days - days_since_prior_order

- Output 1: Current waiting time between orders
- Output 2: Waiting time until the next order
- Output 3: Difference in waiting time between consecutive purchases

In [88]:
con.execute("""WITH waiting_time AS

        (

            SELECT user_id, order_id, order_number, days_since_prior_order,
                   LEAD(days_since_prior_order) OVER(PARTITION BY user_id ORDER BY order_number) as next_order_days
            FROM orders

            )

            SELECT * , (next_order_days - days_since_prior_order) AS days_difference
            FROM waiting_time 

            """).df()

,user_id,order_id,order_number,days_since_prior_order,next_order_days,days_difference
0,55,2080943,1,NaN,9.0,NaN
1,55,2006658,2,9.0,28.0,19.0
2,55,3057843,3,28.0,13.0,-15.0
3,55,3114269,4,13.0,15.0,2.0
4,55,2986671,5,15.0,28.0,13.0
...,...,...,...,...,...,...
3421078,186075,1981535,22,30.0,21.0,-9.0
3421079,186075,2547244,23,21.0,30.0,9.0
3421080,186075,1629710,24,30.0,NaN,NaN
3421081,186111,42452,1,NaN,30.0,NaN


## Observation

The interval between consecutive purchases fluctuated noticeably across customers, with both longer and shorter waiting periods occurring throughout their purchase history. Consecutive orders rarely followed a fixed cadence, indicating that purchasing frequency changed over time rather than remaining consistent. Comparing adjacent purchase intervals exposed shifts in customer buying behavior across successive orders.

In [6]:
### Data Preview — products table

con.execute("""SELECT * FROM products LIMIT 5""").df()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [7]:
### Data Preview — order_products table

con.execute("""SELECT * FROM order_products LIMIT 5""").df()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [8]:
### Data Preview — departments table

con.execute("""SELECT * FROM departments LIMIT 5""").df()

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


# Step 8 — Top Reordered Product in Each Department

## Objective

Identify the most reordered product within every department.

## Task

Return the following columns:

- department_id
- department
- product_id
- product_name
- Total_Reorders
- Department_Reorder_Rank

- Output 1: Total reorders for every product
- Output 2: Product ranking within each department
- Output 3: Highest reordered product from every department

In [27]:
con.execute("""WITH Total_Reorders_Per_Product AS

                (
                
                    SELECT d.department_id, d.department, p.product_id, p.product_name, SUM(op.reordered) as Total_Reorders
                    FROM order_products op
                    LEFT JOIN products p
                    ON op.product_id = p.product_id
                    LEFT JOIN departments d
                    ON d.department_id = p.department_id
                    GROUP BY d.department_id, d.department, p.product_id, p.product_name

                    ),

                Product_Rank_Per_Department AS

                ( 

                    SELECT *, RANK() OVER(PARTITION BY department_id ORDER BY Total_Reorders DESC) as Department_Reorder_Rank
                    FROM Total_Reorders_Per_Product trp
                    
                    )
                            

                    SELECT department_id, department, product_id, product_name, Total_Reorders, 
                            Department_Reorder_Rank
                    FROM Product_Rank_Per_Department
                    WHERE Department_Reorder_Rank = 1

               
                    
                    """).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,department_id,department,product_id,product_name,Total_Reorders,Department_Reorder_Rank
0,13,pantry,31506,Extra Virgin Olive Oil,23954.0,1
1,10,bulk,5161,Dried Mango,7583.0,1
2,2,other,38662,Roasted Almond Butter,2709.0,1
3,6,international,36724,Organic Sea Salt Roasted Seaweed Snacks,6580.0,1
4,12,meat seafood,25890,Boneless Skinless Chicken Breasts,33286.0,1
5,16,dairy eggs,27845,Organic Whole Milk,114510.0,1
6,11,personal care,9047,Premium Epsom Salt,2341.0,1
7,5,alcohol,2120,Sauvignon Blanc,5617.0,1
8,9,dry goods pasta,23375,Marinara Sauce,12521.0,1
9,4,produce,24852,Banana,398609.0,1


## Observation

- Each department has a distinct reorder leader, indicating that customer loyalty is concentrated around a few products.
- Reorder volumes vary considerably across departments, highlighting differences in purchasing behavior across product categories.
- Ranking products within each department makes it straightforward to identify the highest-performing items for comparative analysis.
- The resulting dataset can be directly used for department-level product performance reporting and merchandising decisions.

# Step 8.2 — Top 3 Most Reordered Products Per Department

## Objective

Identify the three most reordered products within every department.

## Task

Identify the **Top 3 most reordered products** in every department.

Return the following columns:

- department_id
- department
- product_id
- product_name
- Total_Reorders
- Department_Reorder_Rank

In [33]:
con.execute("""WITH Total_Reordered_Products AS

                (
                
                    SELECT d.department_id, d.department, p.product_id, p.product_name, SUM(op.reordered) as Total_Reorders
                    FROM order_products op
                    LEFT JOIN products p
                    ON op.product_id = p.product_id
                    LEFT JOIN departments d
                    ON d.department_id = p.department_id
                    GROUP BY d.department_id, d.department, p.product_id, p.product_name

                    ),

                Ranked_Products AS

                (

                    SELECT *, RANK() OVER(PARTITION BY department_id ORDER BY Total_Reorders DESC) as Rank
                    FROM Total_Reordered_Products

                    )

                    SELECT *
                    FROM Ranked_Products 
                    WHERE Rank <=3
                    

                """).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,department_id,department,product_id,product_name,Total_Reorders,Rank
0,21,missing,41149,Organic Riced Cauliflower,4024.0,1
1,21,missing,14010,Organic Mango Yogurt,952.0,2
2,21,missing,7035,Peanut Butter Ice Cream Cup,924.0,3
3,2,other,38662,Roasted Almond Butter,2709.0,1
4,2,other,32115,93/7 Ground Beef,826.0,2
...,...,...,...,...,...,...
58,4,produce,13176,Bag of Organic Bananas,315913.0,2
59,4,produce,21137,Organic Strawberries,205845.0,3
60,20,deli,30489,Original Hummus,51690.0,1
61,20,deli,27344,Uncured Genoa Salami,27372.0,2


## Observation

- Every department contains a small group of products that account for the highest reorder activity.
- Comparing the top three products per department provides better visibility into department-level performance than identifying only the top product.
- Reorder counts decline noticeably after the highest-ranked products, indicating varying levels of customer preference within departments.
- The ranked output can support product assortment analysis, merchandising decisions, and department performance reporting.

# Step 8.3 — Above Department Average Reordered Products

## Objective

Identify products whose reorder count exceeds the average reorder count of their respective department.

## Task

Identify all products whose **Total_Reorders** are **greater than the average Total_Reorders of their respective department**.

Return the following columns:

- department_id
- department
- product_id
- product_name
- Total_Reorders
- Average_Department_Reorders
- Reorder_Difference

In [65]:
con.execute("""WITH Total_Reordered_Products AS

            (   SELECT d.department_id, d.department, p.product_id, p.product_name, SUM(op.reordered) as Total_Reorders
                FROM order_products op
                LEFT JOIN products p
                ON op.product_id = p.product_id
                LEFT JOIN departments d
                ON d.department_id = p.department_id
                GROUP BY d.department_id, d.department, p.product_id, p.product_name
                
                ),
                
                Average_Reorders AS

            (  
        
                 SELECT department_id, AVG(Total_Reorders) AS Average_Department_Reorders
                FROM Total_Reordered_Products
                GROUP BY department_id

                ),

                Joining_AR_With_Diff AS

             (   
                
                SELECT trp.department_id, trp.department, trp.product_id, trp.product_name, trp.Total_Reorders,
                ar.Average_Department_Reorders, (trp.Total_Reorders - ar.Average_Department_Reorders) as Reorder_Difference
                FROM Total_Reordered_Products trp
                INNER JOIN Average_Reorders ar
                ON trp.department_id = ar.department_id
                
                )

                SELECT *
                FROM Joining_AR_With_Diff
                WHERE Total_Reorders > Average_Department_Reorders
                
                """).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,department_id,department,product_id,product_name,Total_Reorders,Average_Department_Reorders,Reorder_Difference
0,1,frozen,12144,Gluten-Free Chicken Nuggets,2915.0,302.443224,2612.556776
1,3,bakery,48183,Flour Tortillas,1655.0,487.591029,1167.408971
2,7,beverages,36316,Lemon Sparkling Water,6613.0,402.909008,6210.090992
3,7,beverages,33198,Sparkling Natural Mineral Water,31500.0,402.909008,31097.090992
4,3,bakery,27750,Sour Batard,1713.0,487.591029,1225.408971
...,...,...,...,...,...,...,...
8737,11,personal care,43175,Optic White Sparkling White Mint Toothpaste,43.0,21.877800,21.122200
8738,21,missing,9638,Bagged Red Potatos,35.0,21.809562,13.190438
8739,11,personal care,3254,Overnight Maxi Wrapped,46.0,21.877800,24.122200
8740,11,personal care,44178,Children's Claritin Non-Drowsy Allergy Relief ...,34.0,21.877800,12.122200


## Observation

- Calculated department-level average reorder counts and joined them back to individual product-level metrics.
- Measured each product's deviation from its department average using a derived difference metric.
- Identified products whose reorder counts exceeded their department's overall average.
- Structured the analysis into separate transformation layers, improving readability and maintainability.

In [67]:
### Data Preview — products table

con.execute("""SELECT * FROM products LIMIT 5""").df()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [68]:
### Data Preview — order_products table

con.execute("""SELECT * FROM order_products LIMIT 5""").df()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [43]:
### Data Preview — orders

con.execute("""SELECT * FROM orders LIMIT 5""").df()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,08,NaN
1,2398795,1,prior,2,3,07,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,07,29.0
4,431534,1,prior,5,4,15,28.0


# Step 8.4.1 — Frequently Purchased Product Pairs

## Objective

Identify which products are most frequently purchased together in the same order.

## Task

Return the following columns:

- product_1_id
- product_1_name
- product_2_id
- product_2_name
- Times_Purchased_Together
- Pair_Rank

Requirements:

- Consider only products purchased within the same order.
- Treat a product pair as unique (A, B should be the same as B, A).
- Count how many orders each unique product pair appears in.
- Rank product pairs based on the number of times they were purchased together (highest first).
- Return the Top 20 most frequently purchased product pairs.

In [8]:
con.execute("""WITH Product_Info AS

             (
                
                SELECT p.product_id, p.product_name, op.order_id
                FROM products p
                LEFT JOIN order_products op
                ON op.product_id = p.product_id

                )

                SELECT pi.product_id as Product_1_id, pi.product_name as Product_1_name, pe.product_id as Product_2_id,
                pe.product_name as Product_2_name, COUNT(*) AS Times_Purchased_Together
                FROM Product_Info pi
                JOIN Product_Info pe
                ON pi.order_id = pe.order_id
                AND pi.product_id < pe.product_id
                WHERE pi.order_id < 100
                GROUP BY pi.product_id, pe.product_id, pi.product_name, pe.product_name
                ORDER BY Times_Purchased_Together DESC

                
                """).df()

,Product_1_id,Product_1_name,Product_2_id,Product_2_name,Times_Purchased_Together
0,24852,Banana,47766,Organic Avocado,3
1,21137,Organic Strawberries,47766,Organic Avocado,3
2,21137,Organic Strawberries,39928,Organic Kiwi,3
3,13176,Bag of Organic Bananas,27966,Organic Raspberries,3
4,21137,Organic Strawberries,24852,Banana,3
...,...,...,...,...,...
5592,8654,Pasta Joy Ready Organic Brown Rice Pasta Penne,30442,Low Fat Vanilla Yogurt,1
5593,20638,Coconut Sugar Pure & Unrefined,30442,Low Fat Vanilla Yogurt,1
5594,4802,"Sesame Oil, Pure",34969,Red Vine Tomato,1
5595,23339,Sweet Corn On The Cob,34969,Red Vine Tomato,1


## Observation

- Identified product combinations by pairing items purchased within the same order using a self join.
- Eliminated duplicate and self-pairs to produce unique product combinations for accurate co-purchase analysis.
- Aggregated product pair occurrences to measure how frequently each combination was purchased together.
- Ranked product combinations by purchase frequency to highlight the strongest product associations.

# Step 8.4.2 — Most Common Companion Product

## Objective

Identify the product that is most frequently purchased together with each product.

## Task

Return the following columns:

- Product_ID
- Product_Name
- Companion_Product_ID
- Companion_Product_Name
- Times_Purchased_Together

Return only the most frequently purchased companion for each product.


In [40]:
con.execute("""WITH Product_Info AS

            (
                SELECT p.product_id, p.product_name, op.order_id
                FROM products p
                LEFT JOIN order_products op
                ON op.product_id = p.product_id
                
                )

                SELECT pi.product_id, pe.product_id as Companion_Product_ID, pi.product_name, 
                pe.product_name as Companion_Product_Name, COUNT(*) AS Times_Purchased_Together,
                DENSE_RANK() OVER(PARTITION BY pi.product_id ORDER BY Times_Purchased_Together DESC) as rank
                FROM Product_Info pi
                JOIN Product_Info pe
                ON pi.order_id = pe.order_id
                AND pi.product_id < pe.product_id
                WHERE pe.order_id < 10000
                GROUP BY pi.product_id, pe.product_id, pi.product_name,pe.product_name
                ORDER BY pi.product_id, Times_Purchased_Together DESC
                                    
                """).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,product_id,Companion_Product_ID,product_name,Companion_Product_Name,Times_Purchased_Together,rank
0,1,16797,Chocolate Sandwich Cookies,Strawberries,3,1
1,1,6184,Chocolate Sandwich Cookies,Clementines,3,1
2,1,43352,Chocolate Sandwich Cookies,Raspberries,2,2
3,1,8048,Chocolate Sandwich Cookies,Packaged Grape Tomatoes,2,2
4,1,9076,Chocolate Sandwich Cookies,Blueberries,2,2
...,...,...,...,...,...,...
549243,49585,49683,Organic Refined Coconut Oil,Cucumber Kirby,1,1
549244,49605,49610,Classic Hummus Family Size,100% Lactose Free Fat Free Milk,1,1
549245,49610,49683,100% Lactose Free Fat Free Milk,Cucumber Kirby,2,1
549246,49628,49683,Yoghurt Blueberry,Cucumber Kirby,2,1


## Observation

- Product co-purchase relationships were identified by generating unique product pairs within the same order through a self join.
- Duplicate pair combinations were eliminated to ensure each product relationship was counted only once.
- Co-purchase frequency was calculated across orders, allowing companion products to be ranked for each primary product.
- The analysis highlighted the strongest product associations that can support recommendation systems and cross-selling opportunities.

# Step 8.4.3 — Customers Sharing the Same Product

## Objective

Identify products that are purchased by multiple different users and determine how many unique user pairs purchased the same product.

## Task

Return the following columns:

- user_1
- user_2
- shared_products_count
- rank

Where:

- `shared_products_count` represents the number of products purchased by both users.
- `rank` orders user pairs from the highest to the lowest number of shared products


In [59]:
con.execute(""" WITH Product_Info as 

      ( 
        SELECT p.product_id, p.product_name, op.order_id, o.user_id
        FROM products p
        LEFT JOIN order_products op
        ON p.product_id = op.product_id
        LEFT JOIN orders o
        ON o.order_id = op.order_id
        
        )
        
        SELECT pi.user_id as user_1, pe.user_id as user_2,
        COUNT(DISTINCT pi.product_id) AS shared_products_count,
        DENSE_RANK() OVER (PARTITION BY pi.user_id ORDER BY shared_products_count DESC) as rank
        FROM Product_Info pi
        JOIN Product_Info pe
        ON pi.product_id = pe.product_id
        AND pi.user_id < pe.user_id
        WHERE pi.order_id < 100
        GROUP BY pi.user_id, pe.user_id
        
                """).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,user_1,user_2,shared_products_count,rank
0,98256,189818,10,1
1,98256,134578,9,2
2,98256,193813,9,2
3,98256,178590,9,2
4,98256,175377,9,2
...,...,...,...,...
2300111,201744,205858,1,5
2300112,201744,205833,1,5
2300113,201744,204286,1,5
2300114,201744,204100,1,5


## Objective

Identify pairs of users who purchased the same products and determine how many products they have in common. Rank user pairs based on the number of shared products.

# Step 8.4.4 — Users with the Highest Number of Shared Products

## Objective

Identify pairs of users who have purchased the highest number of common products.

## Task

Return the following columns:

- user_1
- user_2
- Shared_Products_Count
- Pair_Rank

In [8]:
con.execute("""WITH Product_info AS 

                (
                
                    SELECT p.product_id, p.product_name, op.order_id, o.user_id
                    FROM products p
                    LEFT JOIN order_products op
                    ON p.product_id = op.product_id
                    LEFT JOIN orders o
                    ON o.order_id = op.order_id
                    
                    
                    )
                    
                    SELECT pi.user_id as user_1, pe.user_id as user_2, COUNT(DISTINCT pi.product_id) as Shared_Products_Count,
                    DENSE_RANK() OVER(ORDER BY Shared_Products_Count DESC) AS Pair_Rank
                    FROM Product_info pi
                    JOIN Product_info pe
                    ON pi.product_id = pe.product_id
                    AND pi.user_id < pe.user_id
                    WHERE pi.order_id < 1000
                    GROUP BY pi.user_id, pe.user_id
                    ORDER BY Pair_Rank, pi.user_id
                    
                    
                    
                    
                    
                    
                    
                    """).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,user_1,user_2,Shared_Products_Count,Pair_Rank
0,119437,120222,20,1
1,119437,183416,20,1
2,119437,130393,20,1
3,137716,203482,20,1
4,95063,105501,19,2
...,...,...,...,...
27846866,205970,206059,1,20
27846867,205970,206153,1,20
27846868,205970,206065,1,20
27846869,205970,206000,1,20


## Observation

- Identified pairs of users who purchased the same products by comparing rows within the same dataset using a self join.
- Eliminated duplicate user combinations by applying an ordering condition during the self join.
- Aggregated the number of distinct shared products for each user pair and ranked the pairs based on their overlap.
- Reinforced the use of self joins for relationship analysis between records originating from the same table.

In [9]:
### Data Preview — products table

con.execute("""SELECT * FROM products LIMIT 5""").df()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [10]:
### Data Preview — orders table

con.execute("""SELECT * FROM orders LIMIT 5""").df()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,08,NaN
1,2398795,1,prior,2,3,07,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,07,29.0
4,431534,1,prior,5,4,15,28.0


In [11]:
### Data Preview — order_products table

con.execute("""SELECT * FROM order_products LIMIT 5""").df()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


# Step 8.5 — Consecutive Purchase Behavior Analysis

## Objective

Analyze changes in customer purchasing behavior by comparing consecutive orders over time.

## Task

Return the following columns:

- user_id
- order_id
- order_number
- current_basket_size
- previous_basket_size
- basket_size_change
- purchase_trend

Where:

- `basket_size_change = current_basket_size - previous_basket_size`
- `purchase_trend` should classify each order as:
  - Increased
  - Decreased
  - No Change

In [28]:
con.execute("""WITH Basket_Size AS

            (
                SELECT order_id, COUNT(*) as current_basket_size
                FROM order_products
                GROUP BY order_id

                ),

                Previous_Basket AS

            (

                SELECT bs.order_id, o.order_number, o.user_id, bs.current_basket_size,
                LAG(bs.current_basket_size) OVER(PARTITION BY o.user_id ORDER BY o.order_number) as previous_basket_size
                FROM Basket_Size bs
                LEFT JOIN orders o
                ON bs.order_id = o.order_id
                
                ),

                Basket_Change AS

            (

            
                SELECT user_id, order_id, order_number, current_basket_size, previous_basket_size,
                (current_basket_size - previous_basket_size) AS basket_size_change
                FROM Previous_Basket

                )

                SELECT *, CASE
                            WHEN basket_size_change > 0 THEN 'Increased'
                            WHEN basket_size_change < 0 THEN 'Decreased'
                            ELSE 'No Change' END as purchase_trend
                FROM Basket_Change
                WHERE order_number > 1
                ORDER BY user_id, order_number

""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,user_id,order_id,order_number,current_basket_size,previous_basket_size,basket_size_change,purchase_trend
0,1,2398795,2,6,5,1,Increased
1,1,473747,3,5,6,-1,Decreased
2,1,2254736,4,5,5,0,No Change
3,1,431534,5,8,5,3,Increased
4,1,3367565,6,4,8,-4,Decreased
...,...,...,...,...,...,...,...
3008660,206209,2558525,9,3,12,-9,Decreased
3008661,206209,2266710,10,9,3,6,Increased
3008662,206209,1854736,11,8,9,-1,Decreased
3008663,206209,626363,12,20,8,12,Increased


## Observation

- Calculated the number of products purchased in every order to determine each order's basket size.
- Compared each customer's current order with their immediately previous order using the `LAG()` window function.
- Measured the change in basket size between consecutive orders and classified purchasing behavior as Increased, Decreased, or No Change.
- Demonstrated how window functions enable customer behavior analysis across sequential events without requiring self joins.

# Step 8.6 — Customer Purchase Frequency Segmentation

## Objective

Analyze customer purchasing frequency by measuring the average number of days between consecutive orders and classify customers based on their shopping behavior.

## Task

Return the following columns:

- user_id
- Total_Orders
- Average_Days_Between_Orders
- Purchase_Frequency

Classify customers as:

- Frequent Shopper
- Regular Shopper
- Occasional Shopper

In [29]:
### Data Preview — products table

con.execute("""SELECT * FROM products LIMIT 5""").df()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [30]:
### Data Preview — departments table

con.execute("""SELECT * FROM departments LIMIT 5""").df()

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


In [31]:
### Data Preview — order_products table

con.execute("""SELECT * FROM order_products LIMIT 5""").df()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


# Step 8.6 — Department Leader Gap Analysis

## Objective

Compare every product against the highest reordered product within its own department to measure the performance gap.

## Task

Return the following columns:

- department_id
- department
- product_id
- product_name
- Total_Reorders
- Department_Leader_Reorders
- Reorder_Gap

Where:

- Department_Leader_Reorders = highest reorder count within the department.
- Reorder_Gap = Department_Leader_Reorders - Total_Reorders.

In [50]:
con.execute("""WITH Departmental_Products AS

            (
               SELECT d.department_id, d.department, p.product_id, p.product_name
                FROM departments d
                LEFT JOIN products p
                ON d.department_id = p.department_id
                
                ),
                
                Reorders AS

            (

                
                SELECT dp.department_id, dp.department, dp.product_id, dp.product_name,
                SUM(op.reordered) as Total_Reorders
                FROM Departmental_Products dp
                LEFT JOIN order_products op
                ON dp.product_id = op.product_id
                WHERE op.reordered = 1
                GROUP BY dp.department_id, dp.department, dp.product_id, dp.product_name

                ),

                Department_Leaders AS

            (
                SELECT department_id, department, product_id, product_name, Total_Reorders,
                MAX(Total_Reorders) OVER(PARTITION BY department) as Department_Leader_Reorders
                FROM Reorders
                
                )

                SELECT department_id, department, product_id, product_name, Total_Reorders,
                Department_Leader_Reorders, (Department_Leader_Reorders - Total_Reorders) as Reorder_Gap
                FROM Department_Leaders
                ORDER BY department, Reorder_Gap
                
                
                """).df()

,department_id,department,product_id,product_name,Total_Reorders,Department_Leader_Reorders,Reorder_Gap
0,5,alcohol,2120,Sauvignon Blanc,5617.0,5617.0,0.0
1,5,alcohol,38444,Chardonnay,4230.0,5617.0,1387.0
2,5,alcohol,45190,Vodka,3964.0,5617.0,1653.0
3,5,alcohol,46088,Beer,3915.0,5617.0,1702.0
4,5,alcohol,33065,Cabernet Sauvignon,3687.0,5617.0,1930.0
...,...,...,...,...,...,...,...
45300,19,snacks,25786,Restaurante Style Original Premium Tortilla Chips,1.0,15663.0,15662.0
45301,19,snacks,35674,Angel Food Loaf,1.0,15663.0,15662.0
45302,19,snacks,1829,Mint 72% Raw Cacao Chocolate Bar,1.0,15663.0,15662.0
45303,19,snacks,2951,Chocolate Sea Salt Stars Shortbread Cookies,1.0,15663.0,15662.0


## Observation

Within each department, the highest reordered product establishes a benchmark for comparison. Calculating the reorder gap highlights how far other products trail the department leader, making it easier to identify dominant products and less frequently reordered alternatives.

# Step 8.7 — Running Department Reorder Totals

## Objective

Analyze cumulative reorder performance within each department by tracking how total reorders accumulate from the highest reordered products to the lowest.

## Task

Return the following columns:

- department_id
- department
- product_id
- product_name
- Total_Reorders
- Running_Department_Reorders

Calculate a running total of product reorders within each department. The cumulative total should restart for every department and accumulate from the highest reordered product to the lowest reordered product.

In [5]:
### Data Preview — products table

con.execute("""SELECT * FROM products LIMIT 5""").df()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [6]:
### Data Preview — departments table

con.execute("""SELECT * FROM departments LIMIT 5""").df()

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


In [7]:
### Data Preview — order_products table

con.execute("""SELECT * FROM order_products LIMIT 5""").df()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [67]:
con.execute(""" WITH Departmental_Products AS

             (

                SELECT d.department_id, d.department, p.product_id, p.product_name
                FROM products p
                LEFT JOIN departments d
                ON d.department_id = p.department_id

                ),

                Reorders AS

            (

                SELECT dp.department_id, dp.department, dp.product_id, dp.product_name,
                SUM(op.reordered) as Total_Reorders
                FROM Departmental_Products dp
                LEFT JOIN order_products op
                ON dp.product_id = op.product_id
                WHERE op.reordered = 1
                GROUP BY dp.department_id, dp.department, dp.product_id, dp.product_name

                )

                SELECT department_id, department, product_id, product_name, Total_Reorders,
                SUM(Total_Reorders) OVER(PARTITION BY department ORDER BY Total_Reorders DESC, product_id) as Running_Department_Reorders
                FROM Reorders

                
                """).df()

,department_id,department,product_id,product_name,Total_Reorders,Running_Department_Reorders
0,18,babies,43875,Baby Food Stage 2 Blueberry Pear & Purple Carrot,5685.0,5685.0
1,18,babies,34134,Spinach Peas & Pear Stage 2 Baby Food,5180.0,10865.0
2,18,babies,2611,Gluten Free SpongeBob Spinach Littles,4605.0,15470.0
3,18,babies,3020,Broccoli & Apple Stage 2 Baby Food,4353.0,19823.0
4,18,babies,5491,Stage 1 Apples Sweet Potatoes Pumpkin & Bluebe...,3936.0,23759.0
...,...,...,...,...,...,...
45300,15,canned goods,47703,Sweet Roasted Red Peppers,1.0,488531.0
45301,15,canned goods,47834,Clams Minced,1.0,488532.0
45302,15,canned goods,48365,"Diced Tomatoes with Onion, Celery & Bell Peppers",1.0,488533.0
45303,15,canned goods,48379,Green Enchilada Sauce With Roasted Garlic,1.0,488534.0


# Step 8.8 — Department Product Performance Classification

## Objective

Analyze product reorder performance within each department by combining multiple window functions to identify rankings, cumulative contribution, and performance relative to department leaders.

## Task

Return the following columns:

- department_id
- department
- product_id
- product_name
- Total_Reorders
- Department_Rank
- Running_Department_Reorders
- Department_Leader_Reorders
- Reorder_Percentage_of_Leader

Rank products within each department by total reorders, calculate the cumulative department reorder total, identify the department leader's reorder count, and determine each product's reorder percentage relative to the department leader.

In [18]:
con.execute("""WITH Departmental_Products AS

        (

            SELECT d.department_id, d.department, p.product_id, p.product_name
            FROM products p
            LEFT JOIN departments d
            ON p.department_id = d.department_id

            ),

            Reorders AS (

            SELECT dp.department_id, dp.department, dp.product_id, dp.product_name,
            SUM(op.reordered) as Total_Reorders
            FROM Departmental_Products dp
            LEFT JOIN Order_Products op
            ON dp.product_id = op.product_id
            WHERE reordered = 1
            GROUP BY dp.department_id, dp.department, dp.product_id, dp.product_name


            ),
             
             Departmental_Rank AS

            (
           

            SELECT * , DENSE_RANK() OVER(PARTITION BY department ORDER BY Total_Reorders DESC) AS Department_Rank,
            SUM(Total_Reorders) OVER(PARTITION BY department ORDER BY Total_Reorders DESC, product_id) AS Running_Department_Reorders,
            MAX(Total_Reorders) OVER(PARTITION BY department) AS Department_Leader_Reorders
            FROM Reorders

            )

            SELECT * , ROUND((Total_Reorders * 100.0 / Department_Leader_Reorders),2) AS Reorder_Percentage_of_Leader
            FROM Departmental_Rank
            ORDER BY department, Department_Rank, product_id


            
            """).df()


,department_id,department,product_id,product_name,Total_Reorders,Department_Rank,Running_Department_Reorders,Department_Leader_Reorders,Reorder_Percentage_of_Leader
0,5,alcohol,2120,Sauvignon Blanc,5617.0,1,5617.0,5617.0,100.00
1,5,alcohol,38444,Chardonnay,4230.0,2,9847.0,5617.0,75.31
2,5,alcohol,45190,Vodka,3964.0,3,13811.0,5617.0,70.57
3,5,alcohol,46088,Beer,3915.0,4,17726.0,5617.0,69.70
4,5,alcohol,33065,Cabernet Sauvignon,3687.0,5,21413.0,5617.0,65.64
...,...,...,...,...,...,...,...,...,...
45300,19,snacks,49155,Bacon Cheddar Popcorn Seasoning,1.0,1060,1657969.0,15663.0,0.01
45301,19,snacks,49200,Natural Sunflower Seeds,1.0,1060,1657970.0,15663.0,0.01
45302,19,snacks,49370,Pomegranate Hard Candies,1.0,1060,1657971.0,15663.0,0.01
45303,19,snacks,49400,"Pumpkin Seeds, Heirloom, Organic, With a Bit o...",1.0,1060,1657972.0,15663.0,0.01


## Observation

This query demonstrates how multiple window functions can be combined with Common Table Expressions (CTEs) to produce layered analytical insights. It identifies the highest reordered product within each department, assigns rankings based on reorder frequency, calculates cumulative department reorder totals using a running sum, and measures each product's contribution relative to the department leader through percentage analysis. This pattern is commonly used in business intelligence dashboards and category performance reporting.